# BASTION: New York Electricity Demand Example

This notebook applies **pybastion** to daily average electricity demand for
New York State (2015–2024) and showcases the stochastic-volatility extension.
Section 6.2 of:

> Cho, J. B. & Matteson, D. S. (2026). *BASTION: A Bayesian Framework for
> Trend and Seasonality Decomposition.* arXiv:2601.18052.

## Why this dataset?

- **Weekly seasonality** (lower demand on weekends).
- **Yearly seasonality** (summer A/C + winter heating peaks).
- **Heteroskedastic noise** — volatility is higher in summer/winter.

> *'Higher volatility is observed during the winter and summer months, while
> lower volatility occurs during the spring and fall.'* — Section 6.2

Data source: NYISO via the U.S. Energy Information Administration (EIA, 2024).

## What this notebook covers

1. Load and explore the data.
2. Fit BASTION with K = 7 and K = 365, `obsSV="SV"`.
3. Full decomposition including time-varying volatility.
4. **R BASTION vs pybastion comparison** — posterior-mean agreement on the
   same data using the bundled R reference output.
5. Simulation-average benchmarks (Tables 2 & 3, DGP 3).

In [ ]:
import os, sys
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NOTEBOOK_DIR = os.path.abspath(os.getcwd())
PROJECT_DIR  = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))
OUTPUT_DIR   = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_DIR)
from pybastion import fit_BASTION
from pybastion.datasets import load_NYelectricity

QUICK_MODE = True

if QUICK_MODE:
    nsave, nburn, nskip, nchains = 50, 50, 0, 1
else:
    nsave, nburn, nskip, nchains = 5000, 5000, 4, 1

nstot = nburn + (nskip + 1) * nsave
print(f"pybastion loaded from: {PROJECT_DIR}")
print(f"QUICK_MODE={QUICK_MODE}: {nstot:,} steps × {nchains} chain(s)")

## 1. Load and Explore the Data

Daily average demand (MWh) for NY, 2015-07-01 to 2024-06-30.
Bundled via `pybastion.datasets.load_NYelectricity()`.

In [ ]:
_raw  = load_NYelectricity()
# Raw columns: 'Data.Date' (datetime), 'Demand..MW.' (float)
elec  = _raw.rename(columns={"Data.Date": "date", "Demand..MW.": "demand"}).copy()
elec["date"] = pd.to_datetime(elec["date"])
print(f"Observations : {len(elec)} daily records")
print(f"Date range   : {elec['date'].min().date()} to {elec['date'].max().date()}")
print(f"Demand range : {elec['demand'].min():.0f} to {elec['demand'].max():.0f} MWh")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(elec["date"], elec["demand"], lw=0.6, color="black")
ax.set_xlabel("Date"); ax.set_ylabel("Demand (MWh)")
ax.set_title("NY State Daily Average Electricity Demand 2015–2024")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.tight_layout()
plt.show()

## 2. Fit BASTION with Stochastic Volatility

`obsSV="SV"` activates the Kim-Shephard-Chib (1998) stochastic-volatility
model for the remainder.

| Parameter | Quick | Full | Meaning |
|-----------|-------|------|---------|
| `Ks` | `[7, 365]` | `[7, 365]` | Weekly + yearly seasonalities |
| `Outlier` | `True` | `True` | Horseshoe+ outlier |
| `obsSV` | `"SV"` | `"SV"` | Stochastic volatility remainder |
| `sparse` | `True` | `True` | Sparse precision solver |
| `nsave` | 50 | 5 000 | Saved draws |
| `nburn` | 50 | 5 000 | Burn-in |
| `nskip` | 0 | 4 | Thinning |
| `nchains` | 1 | 1 | Single chain |

In [ ]:
result  = fit_BASTION(
    elec["demand"].values,
    Ks=[7, 365],
    Outlier=True,
    sparse=True,
    obsSV="SV",
    cl=0.95,
    nsave=nsave, nburn=nburn, nskip=nskip, nchains=nchains,
    seed=100,
)
summary = result["summary"]
print("Components:", [k for k in summary if k.endswith("_sum")])
print("Volatility:", "Volatility" in summary)

## 3. Decomposition

- **Trend** — full 9-year period.
- **Weekly seasonality** — 100-day zoom to see weekday/weekend pattern.
- **Yearly seasonality** — summer and winter demand cycles.
- **Volatility** σ_t — posterior-mean observation standard deviation.

In [ ]:
dates  = elec["date"].values
demand = elec["demand"].values

trend_mean = np.asarray(summary["Trend_sum"]["Mean"]).ravel()
trend_lo   = np.asarray(summary["Trend_sum"]["CR_lower"]).ravel()
trend_hi   = np.asarray(summary["Trend_sum"]["CR_upper"]).ravel()
s7_mean    = np.asarray(summary["Seasonal7_sum"]["Mean"]).ravel()
s7_lo      = np.asarray(summary["Seasonal7_sum"]["CR_lower"]).ravel()
s7_hi      = np.asarray(summary["Seasonal7_sum"]["CR_upper"]).ravel()
s365_mean  = np.asarray(summary["Seasonal365_sum"]["Mean"]).ravel()
s365_lo    = np.asarray(summary["Seasonal365_sum"]["CR_lower"]).ravel()
s365_hi    = np.asarray(summary["Seasonal365_sum"]["CR_upper"]).ravel()
volat_mean = np.asarray(summary["Volatility"]["Mean"]).ravel()
volat_lo   = np.asarray(summary["Volatility"]["CR_lower"]).ravel()
volat_hi   = np.asarray(summary["Volatility"]["CR_upper"]).ravel()

zoom    = slice(284, 385)   # ~100-day window
dates_z = dates[zoom]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))

ax = axes[0, 0]
ax.fill_between(dates, trend_lo, trend_hi, color="grey", alpha=0.4)
ax.plot(dates, trend_mean, lw=1.2, color="black")
ax.set_title("Trend"); ax.set_ylabel("MWh")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax = axes[0, 1]
ax.fill_between(dates_z, s7_lo[zoom], s7_hi[zoom], color="grey", alpha=0.4)
ax.plot(dates_z, s7_mean[zoom], lw=1.2, color="black")
ax.set_title("Weekly Seasonality (100-day zoom)"); ax.set_ylabel("MWh")
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

ax = axes[1, 0]
ax.fill_between(dates, s365_lo, s365_hi, color="grey", alpha=0.4)
ax.plot(dates, s365_mean, lw=1.2, color="black")
ax.set_title("Yearly Seasonality"); ax.set_ylabel("MWh")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

ax = axes[1, 1]
ax.fill_between(dates, volat_lo, volat_hi, color="grey", alpha=0.4)
ax.plot(dates, volat_mean, lw=1.2, color="black")
ax.set_title("Volatility σ_t"); ax.set_ylabel("MWh")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

for ax in axes.flatten():
    ax.tick_params(axis="x", labelsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "electricity_decomposition.pdf"), dpi=150,
            bbox_inches="tight")
plt.show()

## 4. Time-Varying Volatility in Detail

A feature unique to BASTION — the posterior mean of σ_t (instantaneous
observation standard deviation) with 95 % credible bands.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.fill_between(dates, volat_lo, volat_hi, color="grey", alpha=0.4, label="95 % CI")
ax.plot(dates, volat_mean, lw=1.2, color="black", label="Posterior mean σ_t")
ax.set_xlabel("Date"); ax.set_ylabel("MWh")
ax.set_title("Time-varying volatility σ_t — NY electricity demand")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=9); fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "electricity_volatility.pdf"), dpi=150,
            bbox_inches="tight")
plt.show()

## 5. R BASTION (paper) vs pybastion — Posterior-Mean Agreement

The R package was used in the paper with the **same data** and the same MCMC
settings (`Ks=[7,365]`, `obsSV="SV"`, `nsave=5000`, `nburn=5000`, `seed=100`).
The bundled reference file `bastion_electric.npz` contains the R posterior
means and credible-interval bounds.  Since both implementations use the same
model and the data is identical, we expect close agreement — discrepancies
come only from MCMC sampling variability and any numerical differences between
R and Python linear-algebra routines.

The table reports:
- **Corr(R, Py)**: Pearson correlation of posterior means.
- **RMSE(R, Py)**: root-mean-square difference of posterior means.
- **Max|Δ|**: maximum absolute difference.

In [ ]:
ref_path = os.path.join(PROJECT_DIR, "pybastion", "data", "reference",
                        "bastion_electric.npz")
if os.path.isfile(ref_path):
    ref = np.load(ref_path)
    rows = []
    for comp, py_arr in [
        ("Trend",        trend_mean),
        ("Weekly (K=7)", s7_mean),
        ("Yearly (K=365)", s365_mean),
        ("Volatility σ_t", volat_mean),
    ]:
        key_prefix = {
            "Trend":        "Trend_sum",
            "Weekly (K=7)": "Seasonal7_sum",
            "Yearly (K=365)": "Seasonal365_sum",
            "Volatility σ_t": "Volatility",
        }[comp]
        r_arr = ref[f"{key_prefix}_Mean"]
        corr  = float(np.corrcoef(r_arr, py_arr)[0, 1])
        rmse  = float(np.sqrt(np.mean((r_arr - py_arr) ** 2)))
        maxd  = float(np.max(np.abs(r_arr - py_arr)))
        rows.append({"Component": comp, "Corr(R, Py)": f"{corr:.6f}",
                     "RMSE(R, Py)": f"{rmse:.2f}", "Max|Δ|": f"{maxd:.2f}"})
    cmp_df = pd.DataFrame(rows).set_index("Component")
    print("R BASTION (paper) vs pybastion — posterior-mean agreement")
    print(cmp_df.to_string())
    if QUICK_MODE:
        print()
        print("(QUICK_MODE — agreement improves substantially with the full chain.)")
else:
    print(f"Reference file not found: {ref_path}")
    print("Run  scripts/convert_rds_to_npz.py  to generate it.")

## 6. Simulation Benchmarks (Paper Tables 2 & 3, DGP 3)

DGP 3 (quadratic trend, Fourier seasonality, stochastic volatility) is the
closest simulation scenario.  Competitors are not in `pybastion`.

| Method | Signal MSE | Trend MSE | Seasonal MSE |
|--------|------------|-----------|-------------|
| TBATS             | 11.307 | 10.509 | 0.900 |
| MSTL              | 3.041  | 0.364  | 2.683 |
| STR               | 0.706  | 0.274  | 0.444 |
| **R BASTION (paper)** | **0.439** | **0.293** | **0.278** |

| Component | STR cov. | R BASTION cov. |
|-----------|----------|----------------|
| Signal       | 0.940 | **0.995** |
| Trend        | 0.812 | **0.929** |
| Seasonality  | 0.847 | **0.981** |

*Source: Tables 2 & 3, DGP 3, Cho & Matteson (2026).*

## 7. In-sample fit — this run

`Signal = Trend + Weekly + Yearly`.

In [ ]:
signal_elec = trend_mean + s7_mean + s365_mean
remainder_e = demand - signal_elec
R2   = float(1 - np.var(remainder_e) / np.var(demand))
rmse = float(np.sqrt(np.mean(remainder_e ** 2)))
avg_vol = float(np.mean(volat_mean))

fit_stats = pd.DataFrame(
    {
        "Statistic": [
            "Signal R² (in-sample)",
            "Remainder RMSE [MWh]",
            "Trend range [MWh]",
            "Weekly season range [MWh]",
            "Yearly season range [MWh]",
            "Mean volatility σ_t [MWh]",
        ],
        "Value": [
            f"{R2:.1%}", f"{rmse:.1f}",
            f"{trend_mean.max() - trend_mean.min():.1f}",
            f"{s7_mean.max() - s7_mean.min():.1f}",
            f"{s365_mean.max() - s365_mean.min():.1f}",
            f"{avg_vol:.1f}",
        ],
    }
).set_index("Statistic")
print("In-sample fit statistics — this run")
print(fit_stats.to_string())